# A2 — Domain model walkthrough (executable)

This notebook drives the A2 model against the **committed frozen data**. It is not a second implementation of
the rules: it imports `scripts/a2_check_invariants.py`, the same code whose raw output is committed at
`data/derived/a2/invariant-checks.md`. What the notebook adds is the interactive path — every number below is
produced here, and the outputs are saved in the file.

How to run: `jupyter nbconvert --to notebook --execute --inplace notebooks/a2-domain-model-walkthrough.ipynb`
(needs `jupyter`; the checks themselves need only the standard library plus `openpyxl` for the workbook import).

In [1]:
import json, os, sys

# find the repository root (the directory holding data/)
root = os.getcwd()
while not os.path.exists(os.path.join(root, "data", "derived", "course-library-snapshot.json")):
    parent = os.path.dirname(root)
    assert parent != root, "repository root not found — run this from inside the repository"
    root = parent
sys.path.insert(0, root)
from scripts import a2_check_invariants as a2      # the one implementation of the rules

LIVE = os.path.join(root, "data/derived/course-library-snapshot.json")
courses = json.load(open(LIVE, encoding="utf-8"))["courses"]
print("repository root :", os.path.basename(root), "(found by walking up from the cwd)")
print("courses loaded  :", len(courses))
print("first course    :", courses[0]["name"], "|", courses[0]["code"])

repository root : project (found by walking up from the cwd)
courses loaded  : 347
first course    : 实用最优化算法 | 180080070100MX018Y


## 1. Reading the schedule fields

`time` and `weeks` are parallel `；`-separated lists — `周三(5-7)；周六(5-7)` against `第2-5,7-17周；第6周`.
Week fields are **interval sets**, never single numbers. The parser returns `None` for anything it cannot read,
and the caller counts those; nothing is ever defaulted to "no clash".

In [2]:
for c in courses[:4]:
    print("%-28s | time=%-28s | weeks=%s" % (c["name"], c["time"], c["weeks"]))
    print("   parsed:", a2.parse_time(c["time"]), a2.parse_weeks(c["weeks"]))

实用最优化算法                      | time=周三(5-7)；周六(5-7)              | weeks=第2-5,7-17周；第6周
   parsed: [(3, 5, 7), (6, 5, 7)] [[(2, 5), (7, 17)], [(6, 6)]]
计算机代数在科学与工程中的应用              | time=周四(10-12)                    | weeks=第2-18周
   parsed: [(4, 10, 12)] [[(2, 18)]]
高等量子力学                       | time=周二(7-8)；周四(7-9)              | weeks=第2-5,7-20周；第2-20周
   parsed: [(2, 7, 8), (4, 7, 9)] [[(2, 5), (7, 20)], [(2, 20)]]
物理中的概率和统计                    | time=周一(10-11)；周三(10-11)；周六(10-11) | weeks=第2-20周；第2-5,7-20周；第6周
   parsed: [(1, 10, 11), (3, 10, 11), (6, 10, 11)] [[(2, 20)], [(2, 5), (7, 20)], [(6, 6)]]


## 2. The decision table, executed

In [3]:
print("%-52s %-8s %-8s %s" % ("case", "expect", "actual", "ok"))
all_ok = True
for name, sa, sb, expected in a2.decision_table_cases():
    got = a2.conflict(sa, sb)
    all_ok &= got == expected
    print("%-52s %-8s %-8s %s" % (name, expected, got, "yes" if got == expected else "NO"))
print()
print("all cases behave as specified:", all_ok)

case                                                 expect   actual   ok
same weekday + periods overlap + weeks overlap       True     True     yes
week ranges disjoint [3,5] vs [6,8]                  False    False    yes
week ranges touching/interleaving [3,5] vs [4,8]     True     True     yes
same weeks, periods apart                            False    False    yes
same weeks and periods, different weekday            False    False    yes
identical section (data duplicated)                  True     True     yes

all cases behave as specified: True


## 3. I1 — conflict structure in the offer set

A timetable is a chosen subset, so overlapping courses in the catalogue are *offers that cannot be combined*.
This is the density of those offers, computed over all 347 scheduled courses.

In [4]:
parsed = {}
for c in courses:
    rows, _ = a2.sessions(c)
    if rows:
        parsed[c["code"]] = rows

codes = sorted(parsed)
pairs = hits = 0
for i, ca in enumerate(codes):
    for cb in codes[i+1:]:
        conflicts = any(a2.conflict(sa, sb) for sa in parsed[ca] for sb in parsed[cb])
        pairs += 1
        hits += conflicts

print("scheduled courses      :", len(codes))
print("course pairs compared  :", pairs)
print("conflicting course pairs:", hits, "(%.1f%%)" % (100.0*hits/pairs))

scheduled courses      : 347
course pairs compared  : 60031
conflicting course pairs: 4757 (7.9%)


## 4. The A1 claim, re-derived from committed data

In [5]:
picks = {}
for needle in ("计算机代数", "数字图像处理与分析", "深度学习"):
    for c in courses:
        if needle in (c.get("name") or "") and c["code"] in parsed:
            picks.setdefault(needle, c)

for needle, c in picks.items():
    print("%-22s %-20s %-14s weeks=%s" % (c["name"], c["code"], c["time"], c["weeks"]))

names = list(picks)
clashing = [(picks[a]["name"], picks[b]["name"])
            for i, a in enumerate(names) for b in names[i+1:]
            if any(a2.conflict(sa, sb) for sa in parsed[picks[a]["code"]] for sb in parsed[picks[b]["code"]])]
print()
print("conflicting pairs among the three:", len(clashing))
for p in clashing:
    print("  -", p[0], "×", p[1])

计算机代数在科学与工程中的应用        180081070100PX001Y   周四(10-12)      weeks=第2-18周
数字图像处理与分析              180093081002P3008Y   周四(10-12)      weeks=第2-19周
深度学习方法与应用              180206085406P3002Y   周四(10-12)      weeks=第2-4,6-15周

conflicting pairs among the three: 3
  - 计算机代数在科学与工程中的应用 × 数字图像处理与分析
  - 计算机代数在科学与工程中的应用 × 深度学习方法与应用
  - 数字图像处理与分析 × 深度学习方法与应用


## 5. I3 — the `capacity = 0` sentinel

The first run reported 40 over-capacity sections. Every one of them had `capacity = 0`. Read literally that is
"40 sections with more students than seats"; read as data it is "40 rows where the capacity was never filled in"
— one shows 86 enrolled. The check keeps the two groups apart so a sentinel can never be laundered into a finding.

In [6]:
real_viol, sentinel, missing = [], [], 0
for c in courses:
    cap, enr = (c.get("capacity") or "").strip(), (c.get("enrolled") or "").strip()
    if not cap or not enr:
        missing += 1
        continue
    cap_i, enr_i = int(float(cap)), int(float(enr))
    if cap_i == 0:
        if enr_i:
            sentinel.append((c["name"], enr_i))
        continue
    if enr_i > cap_i:
        real_viol.append((c["name"], cap_i, enr_i))

print("capacity missing/empty              :", missing)
print("capacity = 0 with enrolment > 0     :", len(sentinel), "(sentinel = 'not set')")
print("enrolled > capacity > 0 (real)      :", len(real_viol))
print()
print("largest sentinel rows:")
for name, enr in sorted(sentinel, key=lambda r: -r[1])[:5]:
    print("   %-24s enrolled=%d capacity=0(not set)" % (name, enr))

capacity missing/empty              : 0
capacity = 0 with enrolment > 0     : 40 (sentinel = 'not set')
enrolled > capacity > 0 (real)      : 0

largest sentinel rows:
   财务与成本管理                  enrolled=86 capacity=0(not set)
   工程经济学                    enrolled=72 capacity=0(not set)
   高级管理学                    enrolled=72 capacity=0(not set)
   工程管理导论-1班                enrolled=72 capacity=0(not set)
   工程哲学                     enrolled=64 capacity=0(not set)


## 6. I2 — planned credits against the programme floors

In [7]:
req = json.load(open(os.path.join(root, "data/s5-programme/degree-requirements.json"), encoding="utf-8"))
m = req["degree_track"]["master"]
plan = json.load(open(os.path.join(root, "data/derived/a2/sample-plan.json"), encoding="utf-8"))["codes"]
by_code = {c["code"]: c for c in courses}

total = sum(float(by_code[c].get("credit") or 0) for c in plan if c in by_code)
print("illustrative plan courses :", len(plan))
print("credits in that plan      :", total)
print("course-learning floor     :", m["course_learning_min_credits"])
print("verdict                   :", "satisfied" if total >= m["course_learning_min_credits"] else "not satisfied (it is a sample)")
print()
print("the programme states floors only — the ceiling half of the invariant has no source, so it is not implemented")

illustrative plan courses : 6
credits in that plan      : 13.5
course-learning floor     : 24
verdict                   : not satisfied (it is a sample)

the programme states floors only — the ceiling half of the invariant has no source, so it is not implemented


## 7. What this walkthrough found (and what it did not)

1. `capacity = 0` is a **sentinel for "not set"**, not a capacity — it invalidated the first version of I3.
2. Week ranges are **interval sets**; the trap is `[3,5]` vs `[4,8]` (clash) against `[3,5]` vs `[6,8]` (no clash).
3. A course normally has several **sections**; clashes are between sections, not courses.
4. The A1 narrative's clash (three courses at 周四第 10-12 节) is **reproducible from committed data** — 3 pairs.
5. The catalogue's 4,757 conflicting course pairs are **density of the offer set**, not invariant violations.

Not settled here: the grab order (needs A3/A4), the decision attached to an over-capacity section, and anything
about spring 2027, whose data does not exist yet.